[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AI-Hypercomputer/maxtext/blob/main/src/maxtext/examples/lora_llama3_demo.ipynb)

# Llama3.1-8B Low-Rank Adaptation (LoRA) Demo


## Overview

This notebook performs LoRA SFT training and evaluation workflow on [OpenAI's GSM8K dataset](https://huggingface.co/datasets/openai/gsm8k).
 The primary goal is to demonstrate the end-to-end process of:
 1. Pre-SFT Evaluation: Calcuating baseline accuracy for the model before training.
 2. Efficient LoRA SFT: Fine-tune the model using MaxText & Tunix SFT trainer.
 3. Post-SFT Evaluation: Re-running the evaluation loop after training to measure the performance gain achieved by LoRA SFT.
 
 This notebook can run on the **public TPU v5e-1**.

## Prerequisites

Before running this notebook, make sure your environment is set up for the method you are using. Follow the [Run MaxText Python Notebooks on TPUs](https://maxtext.readthedocs.io/en/latest/guides/run_python_notebook.html) guide and complete all steps for your chosen method (Google Colab, VS Code, or Local Jupyter Lab) before proceeding.

If you run into issues, refer to the [Common Pitfalls & Debugging](https://maxtext.readthedocs.io/en/latest/guides/run_python_notebook.html#common-pitfalls-debugging) section of the guide.

In [ ]:
try:
  import google.colab
  print("Running the notebook on Google Colab")
  IN_COLAB = True
except ImportError:
    print("Running the notebook on Visual Studio or JupyterLab")
    IN_COLAB = False

## Installation: MaxText and Post training Dependencies

**Running the notebook on Visual Studio or JupyterLab:** Before proceeding, create a virtual environment and install the required post-training dependencies by following `Option 3: Installing [tpu-post-train]` in the [MaxText installation guide](https://maxtext.readthedocs.io/en/latest/install_maxtext.html#from-source). Once the environment is set up, ensure the notebook is running within it.

In [ ]:
if IN_COLAB:
    # Clone the MaxText repository
    !git clone https://github.com/AI-Hypercomputer/maxtext.git
    %cd maxtext

    # Install uv, a fast Python package installer
    !pip install uv
    
    # Install MaxText and post-training dependencies
    !uv pip install -e .[tpu-post-train] --resolution=lowest
    !install_tpu_post_train_extra_deps

**Session restart Instructions for Colab:**
1.  Navigate to the menu at the top of the screen.
2.  Click on **Runtime**.
3.  Select **Restart session** from the dropdown menu.

You will be asked to confirm the action in a pop-up dialog. Click on **Yes**.

## Imports

In [ ]:
import jax
import os
import sys
import subprocess
import transformers

from maxtext.configs import pyconfig
from maxtext.examples.sft_qwen3_demo_utils import evaluate_model, get_test_dataset
from maxtext.integration.tunix.tunix_adapter import TunixMaxTextAdapter
from maxtext.utils.globals import MAXTEXT_PKG_DIR
from maxtext.trainers.post_train.sft import train_sft
from maxtext.checkpoint_conversion.to_huggingface import _get_lora_delta

# Suppress vLLM logging with a severity level below ERROR
os.environ["VLLM_LOGGING_LEVEL"] = "ERROR"
from tunix.rl.rollout import base_rollout
from tunix.rl.rollout.vllm_rollout import VllmRollout

import jax.numpy as jnp

from datetime import datetime
from flax import nnx
from etils import epath

print(f"MaxText installation path: {MAXTEXT_PKG_DIR}")

In [ ]:
if not jax.distributed.is_initialized():
  jax.distributed.initialize()
print(f"JAX version: {jax.__version__}")
print(f"JAX devices: {jax.devices()}")

In [ ]:
if IN_COLAB:
    from huggingface_hub import notebook_login
    notebook_login()
else:
    from huggingface_hub import login
    login()

## Model Configurations

In [ ]:
MODEL_NAME = "llama3.1-8b-Instruct"
TOKENIZER_PATH = "meta-llama/Llama-3.1-8B-Instruct"
tokenizer = transformers.AutoTokenizer.from_pretrained(TOKENIZER_PATH)
# This is the directory where the fine-tuned model checkpoint will be saved
BASE_OUTPUT_DIRECTORY = f"{MAXTEXT_PKG_DIR}/lora_llama3.1-8b_output"

# set the path to the model checkpoint (excluding `/0/items`) or leave empty to download from HuggingFace
MODEL_CHECKPOINT_PATH = ""
if not MODEL_CHECKPOINT_PATH:
   MODEL_CHECKPOINT_PATH = f"{BASE_OUTPUT_DIRECTORY}/llama3.1-8b_checkpoint"
   print("Model checkpoint will be downloaded from HuggingFace at: ",  MODEL_CHECKPOINT_PATH)
   print("Set MODEL_CHECKPOINT_PATH if you do not wish to download the checkpoint.")


RUN_NAME = datetime.now().strftime("%Y-%m-%d-%H-%m-%S")

## Download Llama3.1-8B Model Checkpoint from Hugging Face

In [ ]:
if not epath.Path(MODEL_CHECKPOINT_PATH).exists():
    # Install torch for the conversion script
    print("Installing torch...")
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install",
            "torch", "--index-url", "https://download.pytorch.org/whl/cpu"
        ],
        check=True
    )

    print("Converting checkpoint from HuggingFace...")
    env = os.environ.copy()
    env["JAX_PLATFORMS"] = "cpu"

    subprocess.run(
        [
            sys.executable,
            "-m", "maxtext.checkpoint_conversion.to_maxtext",
            f"{MAXTEXT_PKG_DIR}/configs/base.yml",
            f"model_name={MODEL_NAME}",
            f"base_output_directory={MODEL_CHECKPOINT_PATH}",
            "use_multimodal=false",
            "scan_layers=true",
            "skip_jax_distributed_system=True",
        ],
        check=True,
        env=env
    )
    
    MODEL_CHECKPOINT_PATH = os.path.join(MODEL_CHECKPOINT_PATH, "0/items")
else:
    print(f"Model checkpoint exists at {MODEL_CHECKPOINT_PATH}")

## Dataset Configurations

In [ ]:
DATASET_NAME = "openai/gsm8k"
TRAIN_DATA_SPLIT = "train"
TEST_DATA_SPLIT = "test"
HF_DATA_DIR = "main"
TRAIN_DATA_COLUMNS = ["question", "answer"]
DATA_TEMPLATE_PATH = f"{MAXTEXT_PKG_DIR}/examples/chat_templates/math_qa.json"
if not os.path.exists(DATA_TEMPLATE_PATH):
    raise FileNotFoundError(f"Data template not found: {DATA_TEMPLATE_PATH}")
FORMATTING_FUNC_PATH = "maxtext.input_pipeline.instruction_data_processing.math_qa_formatting"
FORMATTING_FUNC_KWARGS = {"template_path": f"{DATA_TEMPLATE_PATH}"}
NUM_TEST_SAMPLES = 20  # Total number of samples to test
BATCH_SIZE = 1  # Number of test samples to process in a batch

## MaxText Configurations

In [ ]:
%%capture
config = pyconfig.initialize_pydantic(
    [
        "",
        f"{MAXTEXT_PKG_DIR}/configs/post_train/sft.yml",
        f"run_name={RUN_NAME}",
        f"base_output_directory={BASE_OUTPUT_DIRECTORY}",
        f"model_name={MODEL_NAME}",
        f"load_parameters_path={MODEL_CHECKPOINT_PATH}", 
        f"tokenizer_path={TOKENIZER_PATH}",
        f"hf_path={DATASET_NAME}",
        f"train_split={TRAIN_DATA_SPLIT}",
        f"hf_data_dir={HF_DATA_DIR}",
        f"train_data_columns={TRAIN_DATA_COLUMNS}",
        "steps=200",
        "per_device_batch_size=1",
        "max_target_length=512",
        "learning_rate=5e-5", 
        "weight_dtype=bfloat16",
        "dtype=bfloat16",
        f"chat_template_path={DATA_TEMPLATE_PATH}",
        "enable_nnx=true",
        "pure_nnx_decoder=true",
        "scan_layers=true",
        "lora.enable_lora=true",
        "lora.lora_rank=16",
        "lora.lora_alpha=32.0", 
    ]
)

## Initial Setup & Data Preparation

### Create Test Dataset

In [ ]:
run_evaluation = os.environ.get("RUN_EVALUATION", "false").lower() == "true"
run_evaluation = True
if run_evaluation:
    test_dataset = get_test_dataset(config, tokenizer, DATA_TEMPLATE_PATH)
    test_dataset = test_dataset[:NUM_TEST_SAMPLES]
    test_dataset = test_dataset.to_iter_dataset().batch(BATCH_SIZE, drop_remainder=True)
    TOTAL_BATCHES = NUM_TEST_SAMPLES // BATCH_SIZE
    print(
        f"Processing {NUM_TEST_SAMPLES} examples with a batch size of {BATCH_SIZE}. This will result in {TOTAL_BATCHES} total batches for the test run."
    )
else:
    print("Evaluation on test dataset is skipped. Plese set `RUN_EVALUATION=True` to run evaluation.")

### Create LoRA SFT Trainer State

In [ ]:
trainer, mesh = train_sft.setup_trainer_state(config)

### Create vLLM Rollout

In [ ]:
if run_evaluation:
    tunix_model = TunixMaxTextAdapter(trainer.model)
    vllm_rollout = VllmRollout(
        model=tunix_model,
        tokenizer=tokenizer,
        cache_config_or_size=1280,
        mesh=mesh,
        rollout_config=base_rollout.RolloutConfig(
            rollout_vllm_model_version=TOKENIZER_PATH,
            rollout_vllm_hbm_utilization=0.3,
            rollout_vllm_init_with_random_weights=True,
            rollout_vllm_tpu_backend_type="jax",
            data_type=jax.numpy.bfloat16,
            max_tokens_to_generate=512
        ),
    )
else:
    print("Evaluation on test dataset is skipped. Plese set `RUN_EVALUATION=True` to run evaluation.")

## Evaluation before LoRA SFT Training

In [ ]:
if run_evaluation:
    print("Running Pre-SFT Evaluation...")
    score = evaluate_model(test_dataset, vllm_rollout, debug=False)
else:
    print("Evaluation on test dataset is skipped. Plese set `RUN_EVALUATION=True` to run evaluation.")

In [ ]:
if run_evaluation:
    print("========================= Score for PRE-SFT Evaluation =========================")
    print(f"Percentage of test samples where the model produced the correct numerical answer: {score['correct']}%")
    print(
        f"Percentage of test samples where the model produced the numerical answer within 10%: {score['partially_correct']}%"
    )
    print(
        f"Percentage of test samples where the model's output adheres to the expected structure: {score['correct_format']}%"
    )
else:
    print("Evaluation on test dataset is skipped. Plese set `RUN_EVALUATION=True` to run evaluation.")

## LoRA SFT Training

In [ ]:
print("Starting LoRA SFT Training...")
trainer = train_sft.train_model(config, trainer, mesh)
print("LoRA SFT Training Complete!")

## Evaluation after LoRA SFT Training

In [ ]:
if run_evaluation:
    print("Starting weight merge process before evaluation...")

    # --- WHY WE MERGE ---
    # vLLM/VllmRollout initializes with the frozen BASE MODEL and cannot see 
    # the separate LoRA weights in the trainer. We fold LoRA into the base weights 
    # to ensure the evaluation reflects the fine-tuned performance.

    # --- 1. Data Preparation ---
    # Split the NNX model to extract the pure parameter dictionary
    _, state = nnx.split(trainer.model)
    flat_params = state.to_pure_dict()
    scaling = config.lora.lora_alpha / config.lora.lora_rank
    
    # Create an "Official Format" dictionary to align with MaxText's internal naming
    # This ensures _get_lora_delta can correctly map 'kernel' to 'lora_a/b'
    official_dict = {}
    for path, val in jax.tree_util.tree_flatten_with_path(flat_params)[0]:
        # Handle JAX Path objects and strip formatting to get clean string keys
        p_names = [str(p.key if hasattr(p, 'key') else str(p).strip("['']")) for p in path]
        official_key = "params-" + "-".join(p_names)
        official_dict[official_key] = val

    merge_count = 0
    
    # --- 2. Core Merging Logic ---
    # Iterate through parameters and apply LoRA delta using the official MaxText utility
    for off_key, val in official_dict.items():
        if off_key.endswith("-kernel"):
            # Calculate the low-rank delta (A * B * scaling)
            delta = _get_lora_delta(off_key, official_dict, scaling)
            
            if delta is not None:
                # Perform the merge: W_merged = W_base + Delta
                # Includes automatic reshaping for multi-dimensional (Scanned) layers
                merged_val = val + delta.reshape(val.shape) if delta.size == val.size else val + delta
                
                # Navigate back through the nested dictionary to update the value
                original_path = off_key.replace("params-", "").split("-")
                curr = flat_params
                for p in original_path[:-1]:
                    curr = curr[p]
                curr[original_path[-1]] = merged_val
                
                merge_count += 1
                print(f"[Merged] {off_key}")

    # --- 3. Update Model & PROOF OF MERGE ---
    nnx.update(trainer.model, nnx.State(flat_params))
    # Check the merge_count to confirm if LoRA weights have been successfully applied.
    print(f"Weight merging completed! Total modules processed: {merge_count}")

    # Extract the updated state and filter out LoRA variables for vLLM compatibility
    full_updated_state = nnx.state(trainer.model)
    clean_state_dict = {
        k: v for k, v in full_updated_state.items() 
        if "lora" not in str(k)
    }
    clean_state_to_sync = nnx.State(clean_state_dict)

    # --- 4. vLLM Synchronization & Evaluation ---
    print("Syncing merged parameters with vLLM and starting evaluation...")
    vllm_rollout.update_params(clean_state_to_sync)
    score = evaluate_model(test_dataset, vllm_rollout, debug=False)
else:
    print("Evaluation on test dataset is skipped. Please set `RUN_EVALUATION=True` to run evaluation.")

In [ ]:
if run_evaluation:
    print("========================= Score for POST-SFT Evaluation =========================")
    print(f"Percentage of test samples where the model produced the correct numerical answer: {score['correct']}%")
    print(
        f"Percentage of test samples where the model produced the numerical answer within 10%: {score['partially_correct']}%"
    )
    print(
        f"Percentage of test samples where the model's output adheres to the expected structure: {score['correct_format']}%"
    )
else:
    print("Evaluation on test dataset is skipped. Plese set `RUN_EVALUATION=True` to run evaluation.")